In [1]:
import torch
import datetime
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision

# CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # [batch, 1, 28, 28] -> [batch, 32, 28, 28]
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)                           # -> [batch, 32, 14, 14]

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # -> [batch, 64, 14, 14]
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)                           # -> [batch, 64, 7, 7]

        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Data preprocessing
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Model and training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# TensorBoard
# Generate timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
writer = SummaryWriter(f'runs/mnist_experiment_cnn_{timestamp}')
example_images, _ = next(iter(train_loader))
writer.add_graph(model, example_images.to(device))
img_grid = torchvision.utils.make_grid(example_images[:16])
writer.add_image('MNIST_images', img_grid)

# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    correct_train, total_train = 0, 0
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

        step = epoch * len(train_loader) + i
        if i % 100 == 99:
            avg_loss = running_loss / 100
            print(f"[Epoch {epoch+1}, Batch {i+1}] loss: {avg_loss:.3f}")
            writer.add_scalar("training loss", avg_loss, step)
            running_loss = 0.0

        if step % 300 == 0:
            for name, param in model.named_parameters():
                writer.add_histogram(name, param, step)

    # Validation
    model.eval()
    correct_test, total_test = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct_test += (predicted == labels).sum().item()
            total_test += labels.size(0)

    train_acc = correct_train / total_train
    val_acc = correct_test / total_test
    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")
    writer.add_scalars('Accuracy', {'Train': train_acc, 'Validation': val_acc}, epoch)

writer.close()


ModuleNotFoundError: No module named 'tensorboard'

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=runs
# Open http://localhost:6006/ in your browser to view the TensorBoard dashboard.